<a href="https://colab.research.google.com/github/shaiza1iman/loan-approval-prediction/blob/main/notebooks/loan_approval_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Approval Prediction using Logistic Regression

## Phase 1 — Problem Understanding

### Problem Statement

The goal of this project is to build a machine learning model that predicts whether a loan application is likely to be approved or rejected based on applicant information.

### Machine Learning Problem

This is a supervised binary classification problem.

- Target: Loan approval status
- Classes: Approved / Rejected
- Model: Logistic Regression

### Real-World Application

A loan approval prediction system can be used as an educational example of how machine learning can classify loan applications based on applicant characteristics.

> Disclaimer: This project is an educational machine learning demonstration and is not a real-world banking decision system.

## Target and Input Features

### Target Variable

The target variable is the **loan approval status**. The model will predict whether a loan application is approved or rejected.

### Input Features

Input features are the applicant and loan-related information used by the model to make its prediction.

The exact input features will be identified after we inspect the dataset in Phase 2.

## Why Classification?

This project is a classification problem because the model needs to predict one of two possible outcomes:

- Approved
- Rejected

Logistic Regression is appropriate for this project because it is a classification algorithm that can predict the probability of belonging to a class and then make a classification decision.

In [2]:
import pandas as pd

df = pd.read_csv("/content/Loan_Data.csv")

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:
df.shape

In [ ]:
df.info()


In [ ]:
df["Loan_Status"].value_counts()
df.columns.tolist()


## Dataset Information

### Dataset Source

Dataset: Eligibility Prediction for Loan

Source: Kaggle — Devzohaib

The dataset contains information about loan applicants and their loan applications. The target variable is `Loan_Status`, which indicates whether the loan was approved or rejected.

### Dataset Dimensions

- Rows: 614
- Columns: 13

### Column Overview

| Column | Description |
|---|---|
| `Loan_ID` | Unique identifier for the loan application |
| `Gender` | Gender of the applicant |
| `Married` | Marital status of the applicant |
| `Dependents` | Number of dependents |
| `Education` | Education level |
| `Self_Employed` | Whether the applicant is self-employed |
| `ApplicantIncome` | Income of the applicant |
| `CoapplicantIncome` | Income of the co-applicant |
| `LoanAmount` | Requested loan amount |
| `Loan_Amount_Term` | Loan repayment term |
| `Credit_History` | Applicant's credit history information |
| `Property_Area` | Location type of the property |
| `Loan_Status` | Target variable: loan approval status |

### Target Distribution

- Approved (`Y`): 422
- Rejected (`N`): 192

In [3]:
df.isnull().sum()

,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


In [4]:
df.duplicated().sum()


np.int64(0)

In [5]:
X = df.drop(columns=["Loan_Status", "Loan_ID"])
y = df["Loan_Status"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

Features shape: (614, 11)
Target shape: (614,)

Features:
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (491, 11)
X_test: (123, 11)
y_train: (491,)
y_test: (123,)


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify column types
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(exclude=["object"]).columns

# Preprocessing for numerical columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Categorical features:", categorical_features.tolist())
print("Numerical features:", numerical_features.tolist())

Categorical features: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
Numerical features: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']


In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (491, 20)
Processed X_test shape: (123, 20)


In [10]:
import numpy as np

print("Missing values in X_train:", np.isnan(X_train_processed).sum())
print("Missing values in X_test:", np.isnan(X_test_processed).sum())

print("Processed training data type:", type(X_train_processed))

Missing values in X_train: 0
Missing values in X_test: 0
Processed training data type: <class 'numpy.ndarray'>
